In [ ]:
import anndata as ad
import dask.array as da
import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd
import spatialdata
from shapely import affinity
from skimage.draw import polygon2mask
from spatialdata._io.format import SpatialDataContainerFormatV01
from spatialdata.models import PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity, Sequence, Translation, get_transformation, set_transformation

# ROIs of slide No1 are in geojson/No4, those of No2 in geojson/No3
GEOJSON = {"No1": "geojson/No4", "No2": "geojson/No3"}
ROIS = [
    ("No1", "C-ion_Sham", "control1"),
    ("No1", "C-ion_24_h", "Cion8Gy24hr"),
    ("No1", "C-ion_48_h", "Cion8Gy48hr"),
    ("No1", "C-ion_3.5_d", "Cion8Gy3.5d"),
    ("No1", "gamma-ray_Sham", "control2"),
    ("No1", "gamma-ray_24_h", "gamma12Gy24hr"),
    ("No1", "gamma-ray_48_h", "gamma12Gy48hr"),
    ("No1", "gamma-ray_3.5_d", "gamma10Gy3.5d"),
    ("No2", "C-ion_24_h", "Cion8Gy24hr"),
    ("No2", "C-ion_48_h", "Cion8Gy48hr"),
    ("No2", "C-ion_3.5_d", "Cion8Gy84hr"),
    ("No2", "gamma-ray_24_h", "gamma12Gy24hr"),
    ("No2", "gamma-ray_48_h", "gamma12Gy48hr"),
    ("No2", "gamma-ray_3.5_d", "gamma10Gy84hr"),
]
# No2 is placed to the right of No1 so that the two slides do not overlap
SHIFTED = "No2"
GAP = 1000  # um
OUT = "integrated/integrated_mouse.zarr"


def matrix(t):
    return t.to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))


def affine(geom, m):
    return affinity.affine_transform(geom, [m[0, 0], m[0, 1], m[1, 0], m[1, 1], m[0, 2], m[1, 2]])

In [ ]:
rois = pd.DataFrame(ROIS, columns=["slide", "condition", "geojson"])
rois.index = rois["condition"] + "_" + rois["slide"]

tables, cells, nuclei, transcripts, images, to_um, polygons = {}, {}, {}, {}, {}, {}, {}
outside = []
for roi, (slide, condition, name) in rois.iterrows():
    sdata = spatialdata.read_zarr(f"roi_zarrs/{slide}/{condition}.zarr")
    sdata["table"].obs_names = sdata["table"].obs["cell_id"].values
    tables[roi] = sdata["table"]
    cells[roi] = sdata["cell_boundaries"].add_suffix(f"_{roi}", axis=0)
    nuclei[roi] = sdata["nucleus_boundaries"].add_suffix(f"_{roi}", axis=0)
    transcripts[roi] = sdata["transcripts"].assign(roi_id=roi)

    polygon = gpd.read_file(f"{GEOJSON[slide]}/{name}_xenium_explorler_annotation.geojson").geometry[0]

    # pixels outside the ROI are set to 0
    image = spatialdata.get_pyramid_levels(sdata["morphology_focus"], n=0)
    xy = np.asarray(affine(polygon, np.linalg.inv(matrix(get_transformation(image)))).exterior.coords)
    mask = polygon2mask(image.shape[1:], xy[:, ::-1] - 0.5)
    images[roi] = image.copy(data=image.data * da.from_array(mask, chunks=image.data.chunksize[1:]))

    to_um[roi] = get_transformation(sdata["cell_boundaries"]).inverse()
    polygons[roi] = affine(polygon, matrix(to_um[roi]))
    out = list(cells[roi].index[~cells[roi].centroid.within(polygons[roi]).values])
    outside += out
    print(roi, sdata["table"].n_obs, "cells,", len(out), "centroids outside,", f"{1 - mask.mean():.0%} of the image masked")

In [ ]:
moved = rois.index[rois["slide"] == SHIFTED]
others = rois.index[rois["slide"] != SHIFTED]
shift = max(cells[r].total_bounds[2] for r in others) + GAP - min(cells[r].total_bounds[0] for r in moved)
print(f"{SHIFTED} shifted by {shift:.0f} um in x")

for roi in moved:
    for shapes in (cells, nuclei):
        shapes[roi] = shapes[roi].copy()
        shapes[roi]["geometry"] = shapes[roi].translate(xoff=shift)
    transcripts[roi] = transcripts[roi].assign(x=transcripts[roi]["x"] + shift)
    tables[roi].obsm["spatial"] = tables[roi].obsm["spatial"] + [shift, 0]
    polygons[roi] = affinity.translate(polygons[roi], xoff=shift)
roi_shapes = ShapesModel.parse(gpd.GeoDataFrame(geometry=list(polygons.values()), index=list(polygons)))

for roi, image in images.items():
    steps = [get_transformation(image), to_um[roi]]
    if roi in moved:
        steps.append(Translation([shift], axes=("x",)))
    set_transformation(image, Sequence(steps), "global")
images = {f"morphology_focus_{roi}": image for roi, image in images.items()}

In [ ]:
table = ad.concat(tables, label="roi_id", index_unique="_", merge="same", join="inner")
print("genes per ROI:", sorted({t.n_vars for t in tables.values()}), "kept:", table.n_vars)
for column in ["slide", "condition"]:
    table.obs[column] = pd.Categorical(table.obs["roi_id"].map(rois[column]))
# cells in more than one ROI of the same slide, and cells whose centroid is outside the ROI
drop = table.obs[["slide", "cell_id"]].duplicated(keep=False).values | table.obs_names.isin(outside)
print(pd.crosstab(table.obs["roi_id"], drop, colnames=["dropped"]))

dropped = table.obs_names[drop]
table = table[~drop].copy()
table.obs["region"] = "cell_boundaries"
table.obs["cell_key"] = table.obs_names
table = TableModel.parse(table, region="cell_boundaries", region_key="region", instance_key="cell_key")

cell_boundaries = ShapesModel.parse(pd.concat(cells.values()).loc[table.obs_names])
nucleus_boundaries = pd.concat(nuclei.values())
nucleus_boundaries = ShapesModel.parse(nucleus_boundaries[nucleus_boundaries.index.isin(table.obs_names)])
print(pd.crosstab(table.obs["condition"], table.obs["slide"]))

In [ ]:
transcripts = dd.concat(list(transcripts.values()))
key = transcripts["cell_id"].astype("string") + "_" + transcripts["roi_id"].astype("string")
transcripts = PointsModel.parse(
    transcripts[~key.isin(list(dropped))],
    coordinates={"x": "x", "y": "y", "z": "z"},
    feature_key="feature_name",
    instance_key="cell_id",
)

for element in [cell_boundaries, nucleus_boundaries, roi_shapes, transcripts]:
    set_transformation(element, {"global": Identity()}, set_all=True)

In [ ]:
sdata = spatialdata.SpatialData(
    images=images,
    shapes={"cell_boundaries": cell_boundaries, "nucleus_boundaries": nucleus_boundaries, "rois": roi_shapes},
    points={"transcripts": transcripts},
    tables={"table": table},
)
sdata.write(OUT, sdata_formats=[SpatialDataContainerFormatV01()])
spatialdata.read_zarr(OUT)